# 5.13 Derinlemesine: Çekirdek Yoğunluk Tahmini

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/05-sklearn/13-kernel-density-estimation.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 05.13 Kernel Density Estimation

Önceki bölümde Gauss karışım modellerini ele aldık; bunlar kümeleme tahmincisi ile yoğunluk tahmincisinin bir tür hibritidir. Yoğunluk tahmincisinin, $D$ boyutlu bir veri kümesini alıp verinin çekildiği $D$ boyutlu olasılık dağılımının tahminini üreten bir algoritma olduğunu hatırlayın. GMM algoritması yoğunluğu Gauss dağılımlarının ağırlıklı toplamı olarak temsil eder. Çekirdek yoğunluk tahmini (KDE), Gauss karışımı fikrini mantıksal uç noktaya taşır: her nokta için bir Gauss bileşeninden oluşan bir karışım kullanır; sonuçta esasen parametrik olmayan bir yoğunluk tahmincisi elde edilir. Bu bölümde KDE'nin motivasyonunu ve kullanımlarını inceleyeceğiz.

Standart içe aktarmalarla başlayalım:


In [ ]:
# imports_kde.py
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
import numpy as np



## KDE'yi Motive Etmek: Histogramlar

Daha önce belirtildiği gibi yoğunluk tahmincisi, veri kümesini üreten olasılık dağılımını modellemeye çalışan bir algoritmadır. Tek boyutlu veri için muhtemelen tanıdık basit bir yoğunluk tahmincisini biliyorsunuz: histogram. Histogram veriyi ayrık kutulara böler, her kutudaki nokta sayısını sayar ve sonucu sezgisel biçimde görselleştirir.

Örneğin iki normal dağılımdan çekilmiş veri üretelim:


In [ ]:
# make_data_kde.py
def make_data(N, f=0.3, rseed=1):
    rand = np.random.RandomState(rseed)
    x = rand.randn(N)
    x[int(f * N):] += 5
    return x

x = make_data(1000)



Daha önce gördüğümüz gibi standart sayım tabanlı histogram plt.hist ile oluşturulabilir. Histogramın density parametresini belirterek, kutu yükseklikleri sayıları değil olasılık yoğunluğunu yansıtan normalize bir histogram elde ederiz (aşağıdaki şekil):


In [ ]:
# hist_density.py
hist = plt.hist(x, bins=30, density=True)



Eşit kutulama için bu normalleştirme yalnızca y eksenindeki ölçeği değiştirir; göreli yükseklikler sayım histogramıyla esasen aynı kalır. Normalleştirme, histogram altındaki toplam alanın 1 olması için seçilir; histogram fonksiyonunun çıktısıyla doğrulayabiliriz:


In [ ]:
# hist_area_check.py
density, bins, patches = hist
widths = bins[1:] - bins[:-1]
(density * widths).sum()



Histogram yoğunluk tahmincisi olarak kullanıldığında sorunlardan biri, kutu boyutu ve konumu seçiminin niteliksel olarak farklı görünümlere yol açabilmesidir. Örneğin yalnızca 20 noktalık bu verinin farklı kutulama sürümlerine bakarsak, kutuların nasıl çizildiği tamamen farklı bir yorum ortaya çıkarabilir (aşağıdaki şekil):


In [ ]:
# make_data_20.py
x = make_data(20)
bins = np.linspace(-5, 10, 10)



In [ ]:
# hist_bin_compare.py
fig, ax = plt.subplots(1, 2, figsize=(12, 4),
                       sharex=True, sharey=True,
                       subplot_kw={'xlim':(-4, 9),
                                   'ylim':(-0.02, 0.3)})
fig.subplots_adjust(wspace=0.05)
for i, offset in enumerate([0.0, 0.6]):
    ax[i].hist(x, bins=bins + offset, density=True)
    ax[i].plot(x, np.full_like(x, -0.01), '|k',
               markeredgewidth=1)



Solda histogram bunun iki modlu bir dağılım olduğunu net gösterir. Sağda uzun kuyruklu tek modlu bir dağılım görürüz. Önceki kodu görmeden bu iki histogramın aynı veriden yapıldığını tahmin etmek zordur. Histogramların verdiği sezgiye nasıl güvenebiliriz? Nasıl iyileştirebiliriz?


In [ ]:
# block_histogram.py
fig, ax = plt.subplots()
bins = np.arange(-3, 8)
ax.plot(x, np.full_like(x, -0.1), '|k',
        markeredgewidth=1)
for count, edge in zip(*np.histogram(x, bins)):
    for i in range(count):
        ax.add_patch(plt.Rectangle(
            (edge, i), 1, 1, ec='black', alpha=0.5))
ax.set_xlim(-4, 8)
ax.set_ylim(-0.2, 8)



Histogramı, veri kümesindeki her noktanın üzerine bir blok yığdığımız bir blok yığını olarak düşünebiliriz. Bunu doğrudan görelim (aşağıdaki şekil):


In [ ]:
# stacked_blocks.py
x_d = np.linspace(-4, 8, 2000)
density = sum((abs(xi - x_d) < 0.5) for xi in x)

plt.fill_between(x_d, density, alpha=0.5)
plt.plot(x, np.full_like(x, -0.1), '|k', markeredgewidth=1)

plt.axis([-4, 8, -0.2, 8]);



İki kutulamadaki sorun, blok yığınının yüksekliğinin çoğu zaman yakındaki gerçek yoğunluğu değil, kutuların veri noktalarıyla hizalanmasındaki tesadüfleri yansıtmasından kaynaklanır. Noktalar ile blokları arasındaki bu hizasızlık burada görülen zayıf histogram sonuçlarının olası nedenidir. Peki blokları kutularla hizalamak yerine temsil ettikleri noktalarla hizalarsak? Bloklar hizalı olmaz; ancak x ekseni boyunca her konumdaki katkılarını toplayarak sonucu bulabiliriz. Deneyelim (aşağıdaki şekil):


In [ ]:
# gauss_kernel_sum.py
from scipy.stats import norm
x_d = np.linspace(-4, 8, 1000)
density = sum(norm(xi).pdf(x_d) for xi in x)

plt.fill_between(x_d, density, alpha=0.5)
plt.plot(x, np.full_like(x, -0.1), '|k', markeredgewidth=1)

plt.axis([-4, 8, -0.2, 5]);



Sonuç biraz dağınık görünür; ancak standart histogramdan verinin gerçek özelliklerinin çok daha sağlam bir yansımasıdır. Yine de kaba kenarlar estetik değildir ve verinin gerçek özelliklerini yansıtmaz. Yumuşatmak için her konumdaki blokları Gauss gibi düzgün bir fonksiyonla değiştirebiliriz. Her noktada standart normal eğri kullanalım (aşağıdaki şekil):

Her girdi noktasının konumunda Gauss katkısıyla yumuşatılmış bu grafik, veri dağılımının şekline çok daha doğru bir fikir verir ve çok daha az varyansa sahiptir (örneklemedeki farklara daha az duyarlıdır).

Son iki grafikte elde ettiğimiz şey, bir boyutta çekirdek yoğunluk tahminidir: her noktanın konumuna bir "çekirdek" — ilkinde kare veya "tophat", ikincide Gauss — yerleştirdik ve yoğunluk tahmini olarak toplamlarını kullandık. Bu sezgiyle KDE'yi daha ayrıntılı inceleyeceğiz.

## Uygulamada Çekirdek Yoğunluk Tahmini

KDE'nin serbest parametreleri çekirdek (her noktaya yerleştirilen dağılımın şekli) ve çekirdek bant genişliği (her noktadaki çekirdeğin boyutu)dir. Scikit-Learn KDE altı çekirdek destekler (Density Estimation bölümü). Python'da birkaç KDE uygulaması olsa da (SciPy, statsmodels), verimlilik ve esneklik için Scikit-Learn sürümünü tercih ediyorum. sklearn.neighbors.KernelDensity altı çekirdek ve birkaç düzine uzaklık metriğiyle çok boyutlu KDE yapar. Ağaç tabanlı algoritma kullanır; atol/rtol ile hız/doğruluk dengesi kurulabilir. Bant genişliği çapraz doğrulama araçlarıyla belirlenebilir.

Önceki grafiği Scikit-Learn KernelDensity ile çoğaltan basit bir örnekle başlayalım (aşağıdaki şekil):

> **Not**
>


In [ ]:
# sklearn_kde_fit.py
from sklearn.neighbors import KernelDensity

# instantiate and fit the KDE model
kde = KernelDensity(bandwidth=1.0, kernel='gaussian')
kde.fit(x[:, None])

# score_samples returns the log of the probability density
logprob = kde.score_samples(x_d[:, None])

plt.fill_between(x_d, np.exp(logprob), alpha=0.5)
plt.plot(x, np.full_like(x, -0.01), '|k', markeredgewidth=1)
plt.ylim(-0.02, 0.22);



Buradaki sonuç, eğri altındaki alan 1 olacak şekilde normalize edilmiştir.

## Bant Genişliğini Çapraz Doğrulama ile Seçmek

KDE'nin nihai tahmini bant genişliğine oldukça duyarlıdır; yoğunluk tahmininde önyargı–varyans dengesini kontrol eder. Çok dar bant yüksek varyans (aşırı uyum); çok geniş bant yüksek önyargı (yapının yıkanması). Makine öğrenmesinde hiperparametre ayarı genelde çapraz doğrulamayla yapılır. Scikit-Learn KernelDensity grid search ile kullanılabilir. Küçük veri kümesi için leave-one-out CV kullanacağız:

### 🧪 Şimdi deneyin

🧪 
      Basit 1B KDE ile histogram karşılaştırması:
          
      import numpy as np
from sklearn.neighbors import KernelDensity
rng = np.random.default_rng(0)
x = np.concatenate([rng.normal(0, 1, 500), rng.normal(4, 1, 500)])
X = x[:, None]
kde = KernelDensity(bandwidth=0.8).fit(X)
log_dens = kde.score_samples(np.linspace(-3, 8, 100)[:, None])
print("max log-density:", log_dens.max().round(2))


In [ ]:
# kde_gridsearch.py
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import LeaveOneOut

bandwidths = 10 ** np.linspace(-1, 1, 100)
grid = GridSearchCV(KernelDensity(kernel='gaussian'),
                    {'bandwidth': bandwidths},
                    cv=LeaveOneOut())
grid.fit(x[:, None]);



Bant genişliğini maksimize eden seçimi bulabiliriz (varsayılan olarak log-olabilirlik):


In [ ]:
# kde_best_params.py
grid.best_params_



Optimal bant genişliği, daha önce bant genişliği 1.0 olan örnek grafige çok yakındır (scipy.stats.norm varsayılan genişliği).

## Örnek: Tam Naive Olmayan Bayes

Bu örnek KDE ile Bayes üretken sınıflandırmaya bakar ve Scikit-Learn mimarisinde özel bir tahminci oluşturmayı gösterir.

5.5 Naive Bayes Sınıflandırması bölümünde naive Bayes sınıflandırmasını inceledik: her sınıf için basit bir üretken model kurup bu modellerle hızlı sınıflandırıcı oluşturduk. Gauss naive Bayes'te üretken model eksen hizalı basit Gauss'tur. KDE gibi yoğunluk tahmin algoritmasıyla "naive" unsuru kaldırıp her sınıf için daha sofistike üretken modelle aynı sınıflandırmayı yapabiliriz. Hâlâ Bayes sınıflandırmasıdır; ama artık naive değildir.

Genel üretken sınıflandırma yaklaşımı şudur:

1. Eğitim verisini etikete göre bölün.2. Her küme için KDE uydurarak verinin üretken modelini elde edin. Böylece her $(x, y)$ için olabilirlik $P(x~|~y)$ hesaplanabilir.3. Eğitim kümesindeki sınıf örnek sayılarından sınıf önsel $P(y)$ hesaplayın.4. Bilinmeyen $x$ için her sınıfın posterior olasılığı $P(y~|~x) \propto P(x~|~y)P(y)$. Posterioru maksimize eden sınıf, noktaya atanan etikettir.

Algoritma basit ve sezgiseldir; zor kısım bunu grid search ve çapraz doğrulama mimarisinden yararlanmak için Scikit-Learn çerçevesine oturtmaktır. Algoritma kod bloğunda uygulanmıştır; kodu adım adım inceleyeceğiz:


In [ ]:
# kde_classifier.py
from sklearn.base import BaseEstimator, ClassifierMixin

class KDEClassifier(BaseEstimator, ClassifierMixin):
    """Bayesian generative classification based on KDE
    
    Parameters
    ----------
    bandwidth : float
        the kernel bandwidth within each class
    kernel : str
        the kernel name, passed to KernelDensity
    """
    def __init__(self, bandwidth=1.0, kernel='gaussian'):
        self.bandwidth = bandwidth
        self.kernel = kernel
        
    def fit(self, X, y):
        self.classes_ = np.sort(np.unique(y))
        training_sets = [X[y == yi] for yi in self.classes_]
        self.models_ = [KernelDensity(bandwidth=self.bandwidth,
                                      kernel=self.kernel).fit(Xi)
                        for Xi in training_sets]
        self.logpriors_ = [np.log(Xi.shape[0] / X.shape[0])
                           for Xi in training_sets]
        return self
        
    def predict_proba(self, X):
        logprobs = np.array([model.score_samples(X)
                             for model in self.models_]).T
        result = np.exp(logprobs + self.logpriors_)
        return result / result.sum(axis=1, keepdims=True)
        
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), 1)]



### Özel Tahmincinin Anatomisi

Kodu adım adım inceleyelim; temel özellikler:

Scikit-Learn'deki her tahminci bir sınıftır; BaseEstimator ve uygun mixin'den türetmek en uygunudur. BaseEstimator çapraz doğrulama için klonlama mantığını içerir; ClassifierMixin varsayılan score yöntemini tanımlar. Docstring IPython yardımına yakalanır (bkz. 1.1 Yardım ve Dokümantasyon).

Sırada sınıf başlatma yöntemi gelir. Scikit-Learn'de __init__ içinde, geçirilen değerleri self'e atamaktan başka işlem olmaması önemlidir — BaseEstimator klonlama mantığı bunu gerektirir. *args / **kwargs kaçınılmalıdır.

Ardından eğitim verisini işlediğimiz fit yöntemi gelir: eğitim verisindeki benzersiz sınıflar bulunur, her sınıf için KernelDensity eğitilir, sınıf önselleri örnek sayılarından hesaplanır. fit her zaman self döndürmelidir. Kalıcı sonuçlar sondaki alt çizgiyle saklanır (ör. self.logpriors_).

Son olarak yeni veride etiket tahmini: olasılıksal sınıflandırıcı olduğu için önce predict_proba uygulanır; [i,j] girişi örnek i'nin sınıf j üyesi olma posterior olasılığıdır. predict en büyük olasılıklı sınıfı döndürür.

Özel tahmincimizi daha önce gördüğümüz el yazısı rakam sınıflandırmasında deneyelim. Rakamları yükleyip GridSearchCV ile aday bant genişliklerinin çapraz doğrulama skorunu hesaplayacağız (bkz. 5.3 Hiperparametreler):

### 🧪 Şimdi deneyin

🧪 
      Digits verisinde KDE sınıflandırıcı bant genişliği denemesi (kısa):
          
      from sklearn.datasets import load_digits
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KernelDensity
import numpy as np
digits = load_digits()
# Basit tek-sınıf KDE skoru örneği
kde = KernelDensity(bandwidth=1.0).fit(digits.data[digits.target == 0])
print("digit-0 log-lik sample:", kde.score_samples(digits.data[:3]).round(2))


In [ ]:
# kde_digits_grid.py
from sklearn.datasets import load_digits
from sklearn.model_selection import GridSearchCV

digits = load_digits()

grid = GridSearchCV(KDEClassifier(),
                    {'bandwidth': np.logspace(0, 2, 100)})
grid.fit(digits.data, digits.target);



Bant genişliğine karşı çapraz doğrulama skorunu çizebiliriz (aşağıdaki şekil):


In [ ]:
# kde_cv_plot.py
fig, ax = plt.subplots()
ax.semilogx(np.array(grid.cv_results_['param_bandwidth']),
            grid.cv_results_['mean_test_score'])
ax.set(title='KDE Model Performance', ylim=(0, 1),
       xlabel='bandwidth', ylabel='accuracy')
print(f'best param: {grid.best_params_}')
print(f'accuracy = {grid.best_score_}')



KDE sınıflandırıcımız %96'nın üzerinde çapraz doğrulama doğruluğuna ulaşır; naive Bayes sınıflandırıcısı yaklaşık %80 civarındadır:


In [ ]:
# kde_vs_gnb.py
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(), digits.data, digits.target).mean()



Böyle üretken sınıflandırıcının bir faydası, sonuçların yorumlanabilirliğidir: her bilinmeyen örnek için yalnızca olasılıksal sınıflandırma değil, karşılaştırdığımız nokta dağılımının tam modeli elde edilir! SVM ve rastgele orman gibi algoritmaların gizlediği nedenlere sezgisel bir pencere sunar.

Daha ileri gitmek isteyenler için iyileştirme fikirleri:

• Her sınıfta bant genişliğinin bağımsız değişmesine izin verilebilir.• Bant genişlikleri yalnızca tahmin skoruna göre değil, her sınıftaki üretken model olabilirliğine göre optimize edilebilir.• KDE yerine Gauss karışım modelleri kullanan benzer Bayes sınıflandırıcısı kurmak iyi bir alıştırmadır.

> **Not**
>
